# The submission protocol

This notebook implements `docs/evaluation_protocol.md` (section 3.5): the
category scores and the error bars reported for the benchmark, computed so
that an external submitter can reproduce them exactly without exchanging any
files with us.

**Submission requirement.** Five `predictions.csv` files per task, one per
training run. Seeds are the submitter's choice; only the count is enforced.

**Category score.** For a category of $K$ tasks scored over $S$ runs, with
$Y_{k,r}$ the metric of task $k$ under run $r$,

$$\bar{Y} = \frac{1}{K}\sum_{k=1}^{K}\bar{Y}_k,\qquad \bar{Y}_k = \frac{1}{S}\sum_{r=1}^{S} Y_{k,r}.$$

**Error bar.** $\bar{Y} \pm 1.96\,\mathrm{SE}(\bar{Y})$, combining run
variability with test-set sampling noise:

$$\mathrm{SE}(\bar{Y}) = \sqrt{\frac{\sigma_{\mathrm{run}}^{2}}{S} + \sigma_{\mathrm{boot}}^{2}}$$

where $\sigma_{\mathrm{run}}$ is the standard deviation across runs of the
per-run category score and $\sigma_{\mathrm{boot}}$ is the standard deviation
of the category score across $B = 2000$ bootstrap replications. Both are
formed on the *category* score rather than per task, so correlations between
tasks that share a test set are not ignored.

**The replications.** Each one resamples the test set of every *source
dataset* once, and that draw is shared by every task built on that dataset.
Where a dataset provides per-image patient identifiers the resample is at the
patient level, since frames from one patient are correlated.

**Reproducibility.** The resample indices are generated from a hard-coded
seed at run time, so every submission -- past, present and future -- is
bootstrapped on identical draws.

**The gotcha.** The resample is derived from the sorted unique unit IDs of
the benchmark's *test split*, never from the submitted CSV's row order.
Otherwise two submitters who wrote their rows in a different order would get
different draws and the guarantee would evaporate.

In [ ]:
import os
import tempfile
import zlib
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from huggingface_hub import hf_hub_download
from joblib import Parallel, delayed
from sklearn.metrics import f1_score

from notebook_utils import PREDICTIONS_FILENAME, deduplicate_latest, find_runs
from ultrabench.evaluate.tasks import CATEGORIES, TaskSpec, get_task
from ultrabench.core.aggregation import aggregate_sequence_probabilities
from ultrabench.evaluate.segmentation import SegmentationEvaluator

RESAMPLE_SEED = 20260906
"""Frozen at release. Never change: it carries the whole guarantee."""

N_RESAMPLES = 2000
"""Number of test-set resamples drawn per source dataset (B in the formula)."""

REQUIRED_RUNS = 5
"""Runs per task the submission protocol enforces."""

# CONFIDENCE_MULTIPLIER = 1.96
CONFIDENCE_MULTIPLIER = 1.0
"""Normal quantile used for the interval. No t correction, by design."""

PATIENT_COLUMN = "patient"
"""The dataset column holding a per-image patient identifier, where present."""

SWEEPS_DIR = Path(
    os.environ.get(
        "ULTRABENCH_SWEEPS_DIR", "/project/ultrabench2/20260906_results/runs"
    )
)
CACHE_DIR = Path("simplified_protocol_cache_20260908_163207")
N_JOBS = 10

CACHE_DIR.mkdir(exist_ok=True)

MODEL_DISPLAY_NAMES = {
    "deblurmim_convmae": "Deblurring ConvMAE",
    "deblurmim_mae": "Deblurring MAE",
    "dinov3_convnext_base": "DINOv3 ConvNeXt-B",
    "dinov3_convnext_large": "DINOv3 ConvNeXt-L",
    "dinov3_vitb16": "DINOv3 ViT-B",
    "dinov3_vitl16": "DINOv3 ViT-L",
    "echocare": "EchoCare",
    "openus": "OpenUS",
    "samus": "SAMUS",
    "tinyusfm": "TinyUSFM",
    "ultrafedfm": "UltraFedFM",
    "ultrasam": "UltraSAM",
    "urfm_base": "URFM-B",
    "urfm_large": "URFM-L",
    "usfm": "USFM",
    "usfmae": "USF-MAE",
}

# Configure plotting libraries
sns.set_theme(style="whitegrid", context="paper", palette="colorblind", font_scale=0.7)

# Assign a colour to each model so that coloring is consistent with every
# other figure in the paper (see plot_results.ipynb).
MODEL_COLORS = dict(
    zip(
        MODEL_DISPLAY_NAMES.keys(),
        sns.color_palette("husl", n_colors=len(MODEL_DISPLAY_NAMES)),
    )
)

## The frozen resample

`resample_indices` is the function a submitter runs, once per source dataset.
It takes the dataset's unit IDs, sorts them into a canonical order and draws
`B` resamples with replacement.

Three details carry the guarantee:

* `np.unique` sorts, so the draw depends on the *set* of unit IDs, not on the
  order the submitter happened to write their rows in.
* The generator is seeded from the frozen release seed *and* the dataset's
  name, so two datasets that happen to have the same number of units still
  get independent draws, while every task built on one dataset gets the same
  draw.
* `np.random.default_rng` uses PCG64, whose stream NumPy guarantees across
  versions. The legacy `np.random.seed` global state carries no such promise.

In [ ]:
def resample_indices(
    unit_ids: np.ndarray, dataset: str, n_resamples: int = N_RESAMPLES
) -> np.ndarray:
    """Draw the frozen bootstrap resamples of a source dataset's test units.

    Args:
        unit_ids: The unit IDs of the dataset's test split, in any order and
            with any repeats.
        dataset: The name of the source dataset, mixed into the seed so that
            different datasets draw independently.
        n_resamples: The number of resamples to draw.

    Returns:
        An array of shape (n_resamples, n_units) holding the unit IDs drawn
        for each resample.
    """
    units = np.unique(unit_ids)  # sorted, canonical
    seed = [RESAMPLE_SEED, zlib.crc32(dataset.encode())]  # frozen + dataset identity
    rng = np.random.default_rng(seed)  # PCG64: stream is version-stable

    return units[rng.integers(0, len(units), (n_resamples, len(units)))]


def resample_positions(
    units: np.ndarray, dataset: str, n_resamples: int = N_RESAMPLES
) -> np.ndarray:
    """Express the frozen resamples as positions into the sorted unit array.

    Every metric below is computed from per-unit contributions held in
    sorted-unit order, so the draws are more useful as integer positions than
    as the IDs themselves.

    Args:
        units: The sorted unique unit IDs of a dataset's test split.
        dataset: The name of the source dataset.
        n_resamples: The number of resamples to draw.

    Returns:
        An array of shape (n_resamples, n_units) of positions into `units`.

    Raises:
        ValueError: If the positions do not reproduce `resample_indices`.
    """
    draws = resample_indices(units, dataset, n_resamples)
    positions = np.searchsorted(units, draws)

    if not np.array_equal(units[positions], draws):
        raise ValueError("Resample positions do not reproduce the drawn unit IDs.")

    return positions

### The guarantee, checked

Shuffling the submitted row order, and duplicating rows, must leave the
draws byte-identical -- and two datasets of the same size must not collide.

In [ ]:
_units = np.array([f"unit_{index:04d}" for index in range(500)])
_shuffled = np.random.default_rng(1).permutation(_units)

assert np.array_equal(
    resample_indices(_units, "A", 32), resample_indices(_units, "A", 32)
)
assert np.array_equal(
    resample_indices(_units, "A", 32), resample_indices(_shuffled, "A", 32)
)
assert np.array_equal(
    resample_indices(_units, "A", 32),
    resample_indices(np.concatenate([_shuffled, _shuffled]), "A", 32),
)
assert not np.array_equal(
    resample_indices(_units, "A", 32), resample_indices(_units, "B", 32)
)
print("Draws are invariant to row order and repeats, and differ across datasets.")

## The runs

Runs are discovered with `notebook_utils`, so this notebook
walks the sweep directory exactly the way the shipped scoring pipeline does
(`<model>/<task>/shot_<pct>/seed_<seed>/<timestamp>/<head>_<paradigm>/`),
and keeps the latest timestamp per cell.

In [ ]:
TASK_NAMES = sorted(
    {task for category in CATEGORIES.values() for task in category.tasks}
)

runs = [
    run
    for run in deduplicate_latest(find_runs(SWEEPS_DIR))
    if run.shot_pct == 1.0 and run.task in TASK_NAMES
]
runs_df = pd.DataFrame(
    [
        {
            "model": run.model,
            "task": run.task,
            "paradigm": run.paradigm,
            "seed": run.seed,
            "path": run.path,
        }
        for run in runs
    ]
).sort_values(["task", "model", "paradigm", "seed"], ignore_index=True)

runs_per_cell = runs_df.groupby(["model", "task", "paradigm"]).size()
if runs_per_cell.nunique() != 1:
    raise ValueError(f"Uneven run counts per cell:\n{runs_per_cell.value_counts()}")

SEEDS = sorted(runs_df["seed"].unique())
N_RUNS = len(SEEDS)
if int(runs_per_cell.iloc[0]) != N_RUNS:
    raise ValueError(
        "Some cell repeats a seed; the per-run category score is undefined."
    )

print(
    f"{len(runs_df)} runs = {runs_df['model'].nunique()} models"
    f" x {len(TASK_NAMES)} tasks x {runs_df['paradigm'].nunique()} paradigms"
    f" x {N_RUNS} runs"
)
if N_RUNS != REQUIRED_RUNS:
    print(
        f"\nNOTE: the submission protocol requires S={REQUIRED_RUNS} runs per task; "
        f"these results have S={N_RUNS}. The formula is unchanged, but the run "
        f"term sigma_run^2/S is correspondingly wider than a submission's would be."
    )

## Rows, units and source datasets

Three levels have to be kept apart.

* A **row** is what the benchmark scores: an image, or a sequence for the
  three tasks whose label applies to a whole sequence (CAMUS image quality,
  COVID-BLUES, MASLD).
* A **unit** is what the bootstrap resamples: the patient, where the source
  dataset provides a per-image patient identifier, and otherwise the image.
* A **source dataset** owns the draw. Every task built on it is resampled
  with the same draw, so BUS-BRA pathology and histology -- which share a
  test set -- move together across replications.

In [ ]:
# The published filename of each task's ground truth. A dataset repository
# holds every ground-truth CSV in its root, so the name carries the split and,
# for a dataset with more than one subset, the subset. The task suffix is not
# derivable from the registry -- AUL names its two segmentation CSVs after the
# mask while TN3K has only one -- so the mapping is written out here and
# checked against the registry below.
GROUND_TRUTH_FILES = {
    "aul_liver_segmentation": "test.256-ground-truth-segmentation-liver.csv",
    "aul_mass_classification": "test.256-ground-truth-classification.csv",
    "aul_mass_segmentation": "test.256-ground-truth-segmentation-mass.csv",
    "busbra_histology_classification": (
        "test.256-ground-truth-classification-histology.csv"
    ),
    "busbra_pathology_classification": (
        "test.256-ground-truth-classification-pathology.csv"
    ),
    "busbra_segmentation": "test.256-ground-truth-segmentation.csv",
    "camus_classification": "classification-test.256-ground-truth-classification.csv",
    "camus_segmentation": "segmentation-test.256-ground-truth-segmentation.csv",
    "covid_blues_classification": "test.256-ground-truth-classification.csv",
    "fetal_planes_brain_plane_classification": (
        "fetal-brain-test.256-ground-truth-classification.csv"
    ),
    "fetal_planes_plane_classification": (
        "fetal-test.256-ground-truth-classification.csv"
    ),
    "gbcu_classification": "test.256-ground-truth-classification.csv",
    "masld_classification": "test.256-ground-truth-classification.csv",
    "mmotu_classification": "test.256-ground-truth-classification.csv",
    "mmotu_segmentation": "test.256-ground-truth-segmentation.csv",
    "open_kidney_capsule_segmentation": (
        "test.256-ground-truth-segmentation-capsule.csv"
    ),
    "open_kidney_regions_segmentation": (
        "test.256-ground-truth-segmentation-regions.csv"
    ),
    "psfhs_segmentation": "test.256-ground-truth-segmentation.csv",
    "stmus_segmentation": "test.256-ground-truth-segmentation.csv",
    "tn3k_classification": "test.256-ground-truth-classification.csv",
    "tn3k_segmentation": "test.256-ground-truth-segmentation.csv",
}

# A hand-maintained mapping is easy to get wrong, and naming the CSV of the
# wrong split would score resized predictions against full-resolution masks,
# so it is checked rather than trusted.
if set(GROUND_TRUTH_FILES) != set(TASK_NAMES):
    raise ValueError("GROUND_TRUTH_FILES does not name every task exactly once.")

for _task, _filename in GROUND_TRUTH_FILES.items():
    _split = f"test.{get_task(_task).image_size}-"
    if _split not in _filename:
        raise ValueError(
            f"'{_filename}' is not the ground truth of a '{_split}' split."
        )
    if get_task(_task).task_type not in _filename:
        raise ValueError(f"'{_filename}' is not a '{get_task(_task).task_type}' task.")


def ground_truth_csv(task: str) -> Path:
    """Download the ground-truth CSV published with a task's dataset.

    The revision comes from the task registry, so the ground truth can only
    ever be the one belonging to the pinned dataset revision. Revisions before
    0.4 published segmentation masks at full resolution under a `test.256`
    name and cannot be scored; see `docs/evaluation.md`.

    Args:
        task: The name of the task.

    Returns:
        The path of the CSV in the local Hugging Face cache.
    """
    spec = get_task(task)

    return Path(
        hf_hub_download(
            spec.dataset,
            GROUND_TRUTH_FILES[task],
            repo_type="dataset",
            revision=spec.revision,
        )
    )


def load_ground_truth(task: str) -> pd.DataFrame:
    """Read the ground-truth CSV published with a task's dataset.

    Args:
        task: The name of the task.

    Returns:
        The ground truth, with `encoded_pixels` kept as a string so that an
        empty mask does not become NaN.
    """
    return pd.read_csv(ground_truth_csv(task), converters={"encoded_pixels": str})


def row_column(task: str) -> str:
    """Name the ground-truth column identifying a task's scored rows.

    Args:
        task: The name of the task.

    Returns:
        `sequence_id` for sequence-level tasks, `image_id` otherwise.
    """
    return "sequence_id" if get_task(task).sequence_by else "image_id"


def task_rows(task: str) -> np.ndarray:
    """List the sorted unique scored rows of a task's test split.

    Args:
        task: The name of the task.

    Returns:
        The sorted unique row IDs, as strings.
    """
    return np.unique(load_ground_truth(task)[row_column(task)].astype(str).to_numpy())


def load_patient_map(spec: TaskSpec) -> dict[str, str] | None:
    """Map each image of a test split to its patient, if the dataset has one.

    The split is read with only the two columns needed, so the images are
    never decoded. The result is cached as a CSV so that re-running the
    notebook never touches Hugging Face.

    Args:
        spec: The task whose source split to read.

    Returns:
        A mapping of image ID to patient ID, or None if the dataset provides
        no patient identifier.
    """
    subset = spec.subset or "default"
    path = CACHE_DIR / f"patients_{spec.dataset.replace('/', '-')}_{subset}.csv"

    if not path.exists():
        split = datasets.load_dataset(
            spec.dataset,
            name=spec.subset,
            split=f"test.{spec.image_size}",
            revision=spec.revision,
        )
        if PATIENT_COLUMN in split.column_names:
            columns = split.select_columns(["filename", PATIENT_COLUMN])
            table = pd.DataFrame(
                {
                    "image_id": [str(value) for value in columns["filename"]],
                    "patient": [str(value) for value in columns[PATIENT_COLUMN]],
                }
            )
        else:
            table = pd.DataFrame(columns=["image_id", "patient"])
        table.to_csv(path, index=False)

    table = pd.read_csv(path, dtype=str)
    if table.empty:
        return None

    return dict(zip(table["image_id"], table["patient"]))


def task_units(task: str) -> tuple[np.ndarray, str]:
    """Give the resampling unit of each scored row of a task.

    Args:
        task: The name of the task.

    Returns:
        The unit ID of each row in sorted-row order, and whether the units are
        patients or images.

    Raises:
        ValueError: If a sequence spans more than one patient, which would
            make the sequence unassignable to a resampling unit.
    """
    spec = get_task(task)
    rows = ROWS[task]
    patients = load_patient_map(spec)

    if patients is None:
        return rows.copy(), "image"

    if spec.sequence_by is None:
        return np.array([patients[row] for row in rows]), "patient"

    ground_truth = load_ground_truth(task)
    by_sequence = (
        ground_truth["image_id"]
        .astype(str)
        .map(patients)
        .groupby(ground_truth["sequence_id"].astype(str))
        .agg(lambda values: set(values))
    )
    split_sequences = by_sequence[by_sequence.map(len) != 1]
    if len(split_sequences) > 0:
        raise ValueError(
            f"Task '{task}' has {len(split_sequences)} sequence(s) spanning more "
            f"than one patient, starting with '{split_sequences.index[0]}'."
        )

    sequence_patient = by_sequence.map(lambda values: next(iter(values)))

    return np.array([sequence_patient[row] for row in rows]), "patient"


ROWS = {task: task_rows(task) for task in TASK_NAMES}
ROW_UNITS = {task: task_units(task) for task in TASK_NAMES}

DATASET_TASKS: dict[str, list[str]] = {}
for task in TASK_NAMES:
    DATASET_TASKS.setdefault(get_task(task).dataset, []).append(task)

DATASET_UNITS = {
    dataset: np.unique(np.concatenate([ROW_UNITS[task][0] for task in tasks]))
    for dataset, tasks in DATASET_TASKS.items()
}
UNIT_INDEX = {
    task: np.searchsorted(DATASET_UNITS[get_task(task).dataset], ROW_UNITS[task][0])
    for task in TASK_NAMES
}

for task in TASK_NAMES:
    units = DATASET_UNITS[get_task(task).dataset]
    if not np.array_equal(units[UNIT_INDEX[task]], ROW_UNITS[task][0]):
        raise ValueError(f"Task '{task}' has rows outside its dataset's unit list.")

unit_table = pd.DataFrame(
    [
        {
            "task": task,
            "metric": get_task(task).primary_metric,
            "row": row_column(task),
            "n_rows": len(ROWS[task]),
            "unit": ROW_UNITS[task][1],
            "n_task_units": len(np.unique(ROW_UNITS[task][0])),
            "dataset": get_task(task).dataset.removeprefix("UltraBench/"),
            "n_dataset_units": len(DATASET_UNITS[get_task(task).dataset]),
        }
        for task in TASK_NAMES
    ]
)
unit_table

## Per-row contributions

Recomputing a metric on 2000 resamples of every run means recomputing it 4
million times, so each run is reduced once to the per-row quantities the
metric is a function of. Everything after that is linear algebra.

* **Classification** collapses to the confusion matrix, so a run reduces to
  one `(true, predicted)` label pair per row.
* **Segmentation** `positive_dice` is a mean over per-image Dice scores, so a
  run reduces to one Dice value per image.

The reductions reuse the shipped evaluation code -- the same sequence
pooling rule (`aggregate_sequence_probabilities`) and the same RLE decoding
and empty-class convention (`SegmentationEvaluator`) -- so they cannot drift
from the numbers the benchmark reports.

In [ ]:
def _reindex(values: pd.Series, rows: np.ndarray, task: str) -> np.ndarray:
    """Put a per-row series into sorted-row order.

    Args:
        values: The values, indexed by row ID.
        rows: The sorted unique row IDs of the task.
        task: The name of the task, used in the error message.

    Returns:
        The values in the order of `rows`.

    Raises:
        ValueError: If the series does not cover every row of the task.
    """
    values.index = values.index.astype(str)
    missing = np.setdiff1d(rows, values.index.to_numpy())
    if len(missing) > 0:
        raise ValueError(
            f"Task '{task}' has {len(missing)} row(s) with no scored entries, "
            f"starting with '{missing[0]}'."
        )

    return values.reindex(rows).to_numpy()


def classification_contributions(
    task: str, predictions_path: Path
) -> tuple[np.ndarray, np.ndarray]:
    """Reduce one classification run to a label pair per row.

    Args:
        task: The name of the task.
        predictions_path: Path to the run's predictions CSV.

    Returns:
        The ground-truth and predicted labels, in sorted-row order.

    Raises:
        ValueError: If a row is missing a prediction.
    """
    spec = get_task(task)
    rows = ROWS[task]
    merged = load_ground_truth(task).merge(
        pd.read_csv(predictions_path), on="image_id", suffixes=("_gt", "_pred")
    )

    if spec.sequence_by is None:
        indexed = merged.set_index("image_id")
        labels_gt = _reindex(indexed["label_gt"], rows, task)
        labels_pred = _reindex(indexed["label_pred"], rows, task)

        return labels_gt.astype(np.int64), labels_pred.astype(np.int64)

    probability_columns = sorted(
        (c for c in merged.columns if c.startswith("prob_")),
        key=lambda c: int(c.removeprefix("prob_")),
    )
    sequence_ids, probabilities, sequence_labels = aggregate_sequence_probabilities(
        merged["sequence_id"].astype(str).tolist(),
        merged[probability_columns].to_numpy(),
        merged["label_gt"].tolist(),
    )
    pooled = pd.DataFrame(
        {"label_gt": sequence_labels, "label_pred": probabilities.argmax(axis=1)},
        index=pd.Index(sequence_ids, name="sequence_id"),
    )

    return (
        _reindex(pooled["label_gt"], rows, task).astype(np.int64),
        _reindex(pooled["label_pred"], rows, task).astype(np.int64),
    )


def segmentation_contributions(task: str, predictions_path: Path) -> np.ndarray:
    """Reduce one segmentation run to a positive Dice score per image.

    The per-image score is the mean over the foreground classes of the image,
    skipping classes absent from both the ground truth and the prediction. That
    is exactly how `SegmentationEvaluator` builds `positive_dice`, so averaging
    the returned vector reproduces the reported metric.

    Args:
        task: The name of the task.
        predictions_path: Path to the run's predictions CSV.

    Returns:
        The per-image positive Dice scores in sorted-image order. An image
        whose every foreground class is undefined is NaN.
    """
    spec = get_task(task)
    with tempfile.TemporaryDirectory() as temporary:
        evaluator = SegmentationEvaluator(
            ground_truth_path=ground_truth_csv(task),
            predictions_path=predictions_path,
            output_dir=temporary,
            image_height=spec.image_size,
            image_width=spec.image_size,
        )
        evaluator.evaluate()
        per_image = pd.read_csv(Path(temporary) / "per_image_metrics.csv")

    foreground = per_image[per_image["class_id"] != 0]

    return _reindex(
        foreground.groupby("image_id")["dice"].mean(), ROWS[task], task
    ).astype(np.float64)


def run_key(row: pd.Series) -> str:
    """Build the cache key of a run.

    Args:
        row: A row of `runs_df`.

    Returns:
        A key identifying the run within its task.
    """
    return f"{row['model']}|{row['paradigm']}|{row['seed']}"


def build_task_cache(task: str) -> dict[str, np.ndarray]:
    """Reduce every run of a task to its per-row contributions.

    Args:
        task: The name of the task.

    Returns:
        A mapping of cache key to array, ready to be written to an `.npz`.

    Raises:
        ValueError: If two runs disagree on the ground-truth labels.
    """
    task_runs = runs_df[runs_df["task"] == task]
    paths = [row["path"] / PREDICTIONS_FILENAME for _, row in task_runs.iterrows()]
    keys = [run_key(row) for _, row in task_runs.iterrows()]

    if get_task(task).task_type == "segmentation":
        values = Parallel(n_jobs=N_JOBS)(
            delayed(segmentation_contributions)(task, path) for path in paths
        )

        return dict(zip(keys, values))

    pairs = Parallel(n_jobs=N_JOBS)(
        delayed(classification_contributions)(task, path) for path in paths
    )
    cache = {f"pred|{key}": pair[1] for key, pair in zip(keys, pairs)}
    cache["label_gt"] = pairs[0][0]

    for key, (labels_gt, _) in zip(keys, pairs):
        if not np.array_equal(labels_gt, cache["label_gt"]):
            raise ValueError(f"Runs of '{task}' disagree on the ground-truth labels.")

    return cache


def load_task_cache(task: str) -> dict[str, np.ndarray]:
    """Load a task's per-row contributions, building them if necessary.

    Args:
        task: The name of the task.

    Returns:
        A mapping of cache key to array.
    """
    path = CACHE_DIR / f"{task}.npz"
    if not path.exists():
        np.savez_compressed(path, **build_task_cache(task))  # ty:ignore[invalid-argument-type]

    with np.load(path, allow_pickle=False) as archive:
        return {key: archive[key] for key in archive.files}


CONTRIBUTIONS = {}
for task in TASK_NAMES:
    CONTRIBUTIONS[task] = load_task_cache(task)
    print(f"  {task}: {len(ROWS[task])} rows cached")

## Recomputing the metric on a resample

A replication draws units, not rows, so each run's per-row contributions are
first summed up to the units of its source dataset. The multiplicity matrix
$M_D$ -- how many times each of dataset $D$'s units was drawn in each
replication -- is then built once and reused by every task built on $D$.

* Macro F1 is a function of the confusion matrix, and the confusion matrix of
  a replication is $M_D$ times the per-unit cell counts. Classes absent
  from both the resampled truth and the resampled predictions drop out of the
  average, matching `sklearn.metrics.f1_score(average="macro")`.
* Positive Dice is a mean, so a replication gives the ratio estimator
  $\sum_u m_u n_u / \sum_u m_u d_u$ over the per-unit Dice sums. Resampling
  units changes the denominator, which is the point.

In [ ]:
def multiplicity_matrix(dataset: str) -> np.ndarray:
    """Count how many times each of a dataset's units is drawn per replication.

    Args:
        dataset: The name of the source dataset.

    Returns:
        A (N_RESAMPLES, n_units) matrix of draw counts.
    """
    units = DATASET_UNITS[dataset]
    positions = resample_positions(units, dataset)
    n_units = len(units)

    flat = positions + np.arange(N_RESAMPLES)[:, None] * n_units
    counts = np.bincount(flat.ravel(), minlength=N_RESAMPLES * n_units)

    return counts.reshape(N_RESAMPLES, n_units).astype(np.float32)


def macro_f1_from_confusion(confusion: np.ndarray) -> np.ndarray:
    """Compute macro F1 from a stack of confusion matrices.

    A class is averaged over only if it appears in the true or the predicted
    labels, which is the label set `sklearn.metrics.f1_score` uses.

    Args:
        confusion: Confusion matrices of shape (n, n_classes, n_classes),
            indexed as [replication, true, predicted].

    Returns:
        The macro F1 of each confusion matrix.
    """
    true_positives = np.einsum("bii->bi", confusion)
    # 2*TP + FP + FN reduces to the row plus column sums of the class.
    denominator = confusion.sum(axis=2) + confusion.sum(axis=1)
    f1 = np.divide(
        2.0 * true_positives,
        denominator,
        out=np.zeros_like(denominator),
        where=denominator > 0,
    )

    return f1.sum(axis=1) / (denominator > 0).sum(axis=1)


def unit_cell_counts(task: str, key: str) -> np.ndarray:
    """Count each unit's (true, predicted) confusion cells for one run.

    Weighting these rows by how many times each unit was drawn and summing
    gives the confusion matrix of a replication.

    Args:
        task: The name of the task.
        key: The run's cache key.

    Returns:
        A (n_dataset_units, n_classes ** 2) matrix of counts.
    """
    n_classes = get_task(task).num_classes
    cache = CONTRIBUTIONS[task]
    labels_gt = cache["label_gt"]
    labels_pred = cache[f"pred|{key}"]
    n_units = len(DATASET_UNITS[get_task(task).dataset])

    counts = np.zeros((n_units, n_classes * n_classes), dtype=np.float32)
    np.add.at(counts, (UNIT_INDEX[task], labels_gt * n_classes + labels_pred), 1.0)

    return counts


def unit_dice_sums(task: str, key: str) -> tuple[np.ndarray, np.ndarray]:
    """Sum one run's per-image Dice scores within each unit.

    Args:
        task: The name of the task.
        key: The run's cache key.

    Returns:
        The per-unit sum of defined Dice scores and the per-unit count of
        defined images.
    """
    values = CONTRIBUTIONS[task][key]
    defined = np.isfinite(values).astype(np.float64)
    n_units = len(DATASET_UNITS[get_task(task).dataset])

    numerator = np.bincount(
        UNIT_INDEX[task], weights=np.nan_to_num(values) * defined, minlength=n_units
    )
    denominator = np.bincount(UNIT_INDEX[task], weights=defined, minlength=n_units)

    return numerator.astype(np.float32), denominator.astype(np.float32)


def resampled_scores(task: str, key: str, multiplicity: np.ndarray) -> np.ndarray:
    """Recompute a run's task metric on every replication.

    Args:
        task: The name of the task.
        key: The run's cache key.
        multiplicity: The source dataset's multiplicity matrix.

    Returns:
        The metric of the run on each of the N_RESAMPLES replications.
    """
    if get_task(task).task_type == "segmentation":
        numerator, denominator = unit_dice_sums(task, key)

        return (multiplicity @ numerator) / (multiplicity @ denominator)

    n_classes = get_task(task).num_classes
    confusion = (multiplicity @ unit_cell_counts(task, key)).reshape(
        -1, n_classes, n_classes
    )

    return macro_f1_from_confusion(confusion)


def point_score(task: str, key: str) -> float:
    """Compute a run's task metric on the test set itself.

    Args:
        task: The name of the task.
        key: The run's cache key.

    Returns:
        The metric of the run, unresampled.
    """
    if get_task(task).task_type == "segmentation":
        return float(np.nanmean(CONTRIBUTIONS[task][key]))

    n_classes = get_task(task).num_classes
    confusion = unit_cell_counts(task, key).sum(axis=0, dtype=np.float64)

    return float(macro_f1_from_confusion(confusion.reshape(1, n_classes, n_classes))[0])

## Validation

The vectorised metric has to agree with the metric the benchmark actually
reports. Every run's point score is checked against two independent
references: `sklearn` (or `numpy.nanmean` for Dice) applied to the reduced
contributions, and the `test_metrics.csv` the training layer wrote, whose
values are rounded to three or four decimal places.

The two tasks that did *not* share a draw before this notebook resampled at
the dataset level -- the CAMUS pair and the FetalPlanes pair -- are checked
explicitly.

In [ ]:
point_scores = {}
sklearn_gaps = []
reported_gaps = []

for _, row in runs_df.iterrows():
    task, key = row["task"], run_key(row)
    score = point_score(task, key)
    point_scores[(task, key)] = score

    cache = CONTRIBUTIONS[task]
    if get_task(task).task_type == "segmentation":
        reference = float(np.nanmean(cache[key]))
    else:
        reference = f1_score(cache["label_gt"], cache[f"pred|{key}"], average="macro")
    sklearn_gaps.append(abs(score - reference))

    reported = pd.read_csv(row["path"] / "test_metrics.csv").iloc[0]
    reported_gaps.append(abs(score - float(reported[get_task(task).primary_metric])))

print(f"max |ours - sklearn/numpy|      = {max(sklearn_gaps):.3e}")
print(f"max |ours - test_metrics.csv|   = {max(reported_gaps):.3e}")

assert max(sklearn_gaps) < 1e-9
assert max(reported_gaps) < 1e-3

for pair in (
    ("camus_classification", "camus_segmentation"),
    ("fetal_planes_brain_plane_classification", "fetal_planes_plane_classification"),
):
    left, right = (get_task(task).dataset for task in pair)
    assert left == right
    assert np.array_equal(multiplicity_matrix(left), multiplicity_matrix(right))
print("Tasks sharing a source dataset share their draw.")

## The protocol

Every task's metric is recomputed on all `B` replications, then averaged over
the runs. The multiplicity matrix is built once per source dataset and reused
by each of its tasks, which is what makes the replications shared.

In [ ]:
task_terms = []
run_scores = {}
replication_scores = {}

for dataset, dataset_tasks in DATASET_TASKS.items():
    multiplicity = multiplicity_matrix(dataset)

    for task in dataset_tasks:
        task_runs = runs_df[runs_df["task"] == task]

        for (model, paradigm), cell in task_runs.groupby(["model", "paradigm"]):
            keys = [run_key(cell[cell["seed"] == seed].iloc[0]) for seed in SEEDS]
            scores = np.array([point_scores[(task, key)] for key in keys])
            resampled = np.stack(
                [resampled_scores(task, key, multiplicity) for key in keys]
            ).mean(axis=0)

            run_scores[(task, model, paradigm)] = scores
            replication_scores[(task, model, paradigm)] = resampled
            task_terms.append(
                {
                    "task": task,
                    "model": model,
                    "paradigm": paradigm,
                    "mean": scores.mean(),
                    "sigma_k": scores.std(ddof=1),
                    "se_boot_k": resampled.std(ddof=1),
                    "n_rows": len(ROWS[task]),
                    "n_units": len(np.unique(ROW_UNITS[task][0])),
                    "n_runs": len(keys),
                }
            )

    print(f"  {dataset} done ({len(dataset_tasks)} task(s))")

task_terms = pd.DataFrame(task_terms)
task_terms.to_csv("simplified_protocol_task_terms.csv", index=False)
task_terms.head()

Both variance terms are formed on the category score, not per task, so the
correlations induced by a shared test set survive into the error bar.
`se_independent` is the value the same terms would give if the tasks were
treated as independent; it is reported alongside so the size of that effect
is visible.

In [ ]:
def category_intervals(
    runs: dict[tuple, np.ndarray], replications: dict[tuple, np.ndarray]
) -> pd.DataFrame:
    """Combine the per-task scores into a category score and interval.

    Args:
        runs: The per-run task scores, keyed by (task, model, paradigm).
        replications: The run-averaged per-replication task scores, keyed the
            same way.

    Returns:
        One row per (category, model, paradigm) with the category mean, its
        standard error, the 95% interval and the two variance terms.

    Raises:
        ValueError: If a model does not cover every task of a category.
    """
    rows = []
    cells = sorted({(model, paradigm) for _, model, paradigm in runs})

    for name, category in CATEGORIES.items():
        for model, paradigm in cells:
            keys = [(task, model, paradigm) for task in category.tasks]
            missing = [key[0] for key in keys if key not in runs]
            if missing:
                raise ValueError(
                    f"{model}/{paradigm} is missing {len(missing)} task(s) of "
                    f"'{name}', starting with '{missing[0]}'."
                )

            per_run = np.stack([runs[key] for key in keys])  # K x S
            per_replication = np.stack([replications[key] for key in keys])  # K x B
            k, n_runs = per_run.shape

            sigma_run = per_run.mean(axis=0).std(ddof=1)
            sigma_boot = per_replication.mean(axis=0).std(ddof=1)
            standard_error = np.sqrt(sigma_run**2 / n_runs + sigma_boot**2)
            mean = per_run.mean()

            independent = (
                np.sqrt(
                    (per_run.std(axis=1, ddof=1) ** 2 / n_runs).sum()
                    + (per_replication.std(axis=1, ddof=1) ** 2).sum()
                )
                / k
            )

            rows.append(
                {
                    "category": name,
                    "display_name": category.display_name,
                    "model": model,
                    "paradigm": paradigm,
                    "mean": mean,
                    "se": standard_error,
                    "ci_low": mean - CONFIDENCE_MULTIPLIER * standard_error,
                    "ci_high": mean + CONFIDENCE_MULTIPLIER * standard_error,
                    "sigma_run": sigma_run,
                    "sigma_boot": sigma_boot,
                    "se_independent": independent,
                    "k_tasks": k,
                }
            )

    return pd.DataFrame(rows)


intervals = category_intervals(run_scores, replication_scores)
intervals.to_csv("simplified_protocol_category_intervals.csv", index=False)
intervals.head()

## Pretraining overlap

Several of these encoders were pretrained on data drawn from the very
datasets the benchmark scores them on, so a category score is not comparable
across models without saying so. The table below records, per model and per
category, which of the category's tasks overlap that model's pretraining
data. It mirrors the table in `plot_results.ipynb`, rekeyed to the category
names in `CATEGORIES`.

`None` means the model's pretraining sources are not publicly documented, as
opposed to `[]`, which is an asserted "no known overlap". The distinction
matters: an undocumented model cannot be called clean, only unknown, and it
is marked `(?)` in the figure rather than `(0)`.


In [ ]:
# Per-model record of which benchmark tasks in each category overlap with the
# model's pretraining data. `None` means the model's pretraining data sources
# are not publicly documented/known, as opposed to `[]` which is an asserted
# "no known overlap" for that category. Populate the inner lists by hand from
# each model's paper/model card.
CONTAMINATION: dict[str, dict[str, list[str]] | None] = {
    "deblurmim_convmae": {
        "pathology_classification": [],
        "anatomy_classification": [],
        "pathology_segmentation": [],
        "anatomy_segmentation": [],
    },
    "deblurmim_mae": {
        "pathology_classification": [],
        "anatomy_classification": [],
        "pathology_segmentation": [],
        "anatomy_segmentation": [],
    },
    "dinov3_convnext_base": {
        "pathology_classification": [],
        "anatomy_classification": [],
        "pathology_segmentation": [],
        "anatomy_segmentation": [],
    },
    "dinov3_convnext_large": {
        "pathology_classification": [],
        "anatomy_classification": [],
        "pathology_segmentation": [],
        "anatomy_segmentation": [],
    },
    "dinov3_vitb16": {
        "pathology_classification": [],
        "anatomy_classification": [],
        "pathology_segmentation": [],
        "anatomy_segmentation": [],
    },
    "dinov3_vitl16": {
        "pathology_classification": [],
        "anatomy_classification": [],
        "pathology_segmentation": [],
        "anatomy_segmentation": [],
    },
    "echocare": None,
    "openus": {
        "pathology_classification": [
            "aul_mass_classification",
            "mmotu_classification",
        ],
        "anatomy_classification": ["camus_classification"],
        "pathology_segmentation": ["aul_mass_segmentation", "mmotu_segmentation"],
        "anatomy_segmentation": [
            "aul_liver_segmentation",
            "camus_segmentation",
            "stmus_segmentation",
        ],
    },
    "samus": {
        "pathology_classification": ["tn3k_classification"],
        "anatomy_classification": ["camus_classification"],
        "pathology_segmentation": ["tn3k_segmentation"],
        "anatomy_segmentation": ["camus_segmentation"],
    },
    "tinyusfm": None,
    "ultrafedfm": {
        "pathology_classification": [],
        "anatomy_classification": [
            "camus_classification",
            "fetal_planes_brain_plane_classification",
            "fetal_planes_plane_classification",
        ],
        "pathology_segmentation": [],
        "anatomy_segmentation": ["camus_segmentation"],
    },
    "ultrasam": {
        "pathology_classification": [
            "aul_mass_classification",
            "tn3k_classification",
        ],
        "anatomy_classification": ["camus_classification"],
        "pathology_segmentation": ["aul_mass_segmentation", "tn3k_segmentation"],
        "anatomy_segmentation": [
            "aul_liver_segmentation",
            "camus_segmentation",
            "open_kidney_capsule_segmentation",
            "open_kidney_regions_segmentation",
            "psfhs_segmentation",
            "stmus_segmentation",
        ],
    },
    "urfm_base": {
        "pathology_classification": [],
        "anatomy_classification": [
            "camus_classification",
            "fetal_planes_brain_plane_classification",
            "fetal_planes_plane_classification",
        ],
        "pathology_segmentation": [],
        "anatomy_segmentation": ["camus_segmentation"],
    },
    "urfm_large": {
        "pathology_classification": [],
        "anatomy_classification": [
            "camus_classification",
            "fetal_planes_brain_plane_classification",
            "fetal_planes_plane_classification",
        ],
        "pathology_segmentation": [],
        "anatomy_segmentation": ["camus_segmentation"],
    },
    "usfm": None,
    "usfmae": {
        "pathology_classification": [
            "aul_mass_classification",
            "covid_blues_classification",
            "tn3k_classification",
        ],
        "anatomy_classification": [
            "camus_classification",
            "fetal_planes_brain_plane_classification",
            "fetal_planes_plane_classification",
        ],
        "pathology_segmentation": ["aul_mass_segmentation", "tn3k_segmentation"],
        "anatomy_segmentation": [
            "aul_liver_segmentation",
            "camus_segmentation",
            "open_kidney_capsule_segmentation",
            "open_kidney_regions_segmentation",
            "psfhs_segmentation",
            "stmus_segmentation",
        ],
    },
}

# A hand-maintained table is easy to get wrong in a way no figure would show,
# so check it against the task registry rather than trusting it.
missing_models = sorted(set(MODEL_DISPLAY_NAMES) - set(CONTAMINATION))
if missing_models:
    raise ValueError(f"No pretraining record for {missing_models}.")

for _model, _record in CONTAMINATION.items():
    if _record is None:
        continue
    if set(_record) != set(CATEGORIES):
        raise ValueError(f"'{_model}' does not name every category exactly once.")
    for _category, _tasks in _record.items():
        _stray = sorted(set(_tasks) - set(CATEGORIES[_category].tasks))
        if _stray:
            raise ValueError(
                f"'{_model}' lists {_stray} under '{_category}', which holds no "
                f"such task."
            )


def contamination_label(model: str, category: str) -> str:
    """Build the model's X-axis tick label, annotated with its overlap.

    Args:
        model: The internal model key.
        category: The name of the category being plotted.

    Returns:
        `"<display name> (?)"` if the model's pretraining data is undocumented,
        otherwise `"<display name> (<n>)"` with `n` the number of the
        category's tasks that overlap the model's pretraining data.
    """
    display = MODEL_DISPLAY_NAMES[model]
    record = CONTAMINATION.get(model)
    if record is None:
        return f"{display} (?)"

    return f"{display} ({len(record[category])})"


def is_clean(model: str, category: str) -> bool | None:
    """Whether a model has zero known pretraining overlap in a category.

    Args:
        model: The internal model key.
        category: The name of the category being checked.

    Returns:
        True if the model has no known overlapping task in the category, False
        if it has at least one, or None if the model's pretraining data is
        undocumented.
    """
    record = CONTAMINATION.get(model)
    if record is None:
        return None

    return len(record[category]) == 0

## The figure

Top row classification, bottom row segmentation. Whiskers span the full
$\pm 1.96\,\mathrm{SE}$ interval, not one standard error, so they read
directly as a 95% interval.

Each model's tick label carries its pretraining overlap with the category:
`(n)` counts the category's tasks that appear in the model's pretraining
data, and `(?)` marks a model whose pretraining sources are undocumented.
The label of the best full fine-tuning score among the models with zero
known overlap is bolded, since that is the highest score in the panel that
can be read at face value.


In [ ]:
def plot_category_panel(
    ax: plt.Axes, category: str, intervals: pd.DataFrame, title: str
) -> list[str]:
    """Plot the score and interval of every model in one category.

    Models are ordered along the X-axis in descending order of their mean full
    fine-tuning score, matching the convention in plot_results.ipynb. Tick
    labels are annotated with the model's pretraining overlap, and the best
    model with no known overlap is bolded.

    Args:
        ax: The axes to draw the panel on.
        category: The name of the category to plot.
        intervals: The category-level table from `category_intervals`.
        title: The panel title.

    Returns:
        The models in the order they were plotted.
    """
    selected = intervals[intervals["category"] == category]
    full = selected[selected["paradigm"] == "full_finetuning"].sort_values(
        "mean", ascending=False
    )
    head_only = selected[selected["paradigm"] == "head_only"].set_index("model")
    model_order = full["model"].tolist()

    for position, model in enumerate(model_order):
        for frame, marker, alpha in (
            (full.set_index("model"), "o", 1.0),
            (head_only, "D", 0.45),
        ):
            if model not in frame.index:
                continue
            row = frame.loc[model]
            ax.errorbar(
                position,
                row["mean"],
                yerr=CONFIDENCE_MULTIPLIER * row["se"],
                color=MODEL_COLORS[model],
                alpha=alpha,
                marker=marker,
                markersize=3,
                elinewidth=0.8,
                capsize=1.5,
            )

    ax.set_title(title)
    ax.set_xlim(-0.5, len(model_order) - 0.5)
    ax.set_xticks(
        range(len(model_order)),
        [contamination_label(model, category) for model in model_order],
        rotation=45,
        ha="right",
    )
    ax.set_xlabel("")

    # Bold the best-performing (full fine-tuning) model with zero known
    # pretraining overlap in this category, if any. `model_order` is already
    # sorted by that score, so the first clean model is the best one.
    clean_models = [model for model in model_order if is_clean(model, category)]
    if clean_models:
        ax.get_xticklabels()[model_order.index(clean_models[0])].set_fontweight("bold")

    return model_order


fig, axes = plt.subplots(2, 2, figsize=(6.5, 4.25))

plot_category_panel(
    axes[0, 0], "pathology_classification", intervals, "Pathological Classification"
)
plot_category_panel(
    axes[0, 1], "anatomy_classification", intervals, "Non-Pathological Classification"
)
plot_category_panel(
    axes[1, 0], "pathology_segmentation", intervals, "Pathological Segmentation"
)
plot_category_panel(
    axes[1, 1], "anatomy_segmentation", intervals, "Anatomical Segmentation"
)

axes[0, 0].set_ylabel("Macro F1")
axes[1, 0].set_ylabel("Positive Dice")

legend_handles = [
    plt.Line2D(
        [],
        [],
        color="0.3",
        linestyle="none",
        marker=marker,
        markersize=3,
        alpha=alpha,
        label=label,
    )
    for marker, alpha, label in (
        ("o", 1.0, "Full fine-tuning"),
        ("D", 0.45, "Head only"),
    )
]
# fig.legend(handles=legend_handles, loc="lower center", ncol=2, frameon=False)
# fig.suptitle(
#     f"95% interval, B={N_RESAMPLES} test resamples, S={N_RUNS} runs"
# )

# Increase alpha of grid lines for better visibility
for ax in axes.flatten():
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("simplified_protocol.pdf")
plt.savefig("simplified_protocol.png", dpi=300)
plt.show()

## Where the width comes from

The two terms behave very differently. $\sigma_{\mathrm{run}}$ shrinks as the
submitter trains more runs; $\sigma_{\mathrm{boot}}$ is a property of the
benchmark's test cohorts and no amount of training will move it.

`se / se_independent` is what treating the tasks as independent would have
cost. It exceeds one wherever a category's tasks move together across
replications, which is what sharing the draw per source dataset produces.

One honest caveat while $S$ is small: forming $\sigma_{\mathrm{run}}$ on the
category score leaves it $S - 1$ degrees of freedom, where combining per-task
standard deviations would have pooled $K(S - 1)$. That is the price of not
assuming independence, and it is why the ratio scatters around one rather
than sitting above it at $S = 3$. It shrinks at $S = 5$.

In [ ]:
SE_COLUMNS = ["se", "sigma_run", "sigma_boot", "se_independent"]

decomposition = (
    intervals[intervals["paradigm"] == "full_finetuning"]
    .groupby("display_name")[SE_COLUMNS]
    .median()
    .round(4)
)
decomposition["se_over_independent"] = (
    decomposition["se"] / decomposition["se_independent"]
).round(3)
decomposition

In [ ]:
correlations = []
for name, category in CATEGORIES.items():
    for model, paradigm in sorted({(m, p) for _, m, p in replication_scores}):
        stacked = np.stack(
            [replication_scores[(task, model, paradigm)] for task in category.tasks]
        )
        matrix = np.corrcoef(stacked)
        off_diagonal = matrix[~np.eye(len(matrix), dtype=bool)]
        correlations.append(
            {
                "display_name": category.display_name,
                "paradigm": paradigm,
                "mean_pairwise_r": off_diagonal.mean(),
                "max_pairwise_r": off_diagonal.max(),
            }
        )

pd.DataFrame(correlations).groupby("display_name")[
    ["mean_pairwise_r", "max_pairwise_r"]
].median().round(3)

## Appendix: what to pin in the paper

Everything a submitter needs to reproduce these draws, and the caveats worth
stating explicitly.

In [ ]:
print(f"RESAMPLE_SEED            {RESAMPLE_SEED}")
print(f"Replications B           {N_RESAMPLES}")
print(f"Runs per task S          {N_RUNS} (protocol requires {REQUIRED_RUNS})")
print("Generator                np.random.default_rng (PCG64)")
print("Seed entropy             [RESAMPLE_SEED, crc32(dataset name)]")
print(f"NumPy version            {np.__version__}")
print(f"Interval                 mean +/- {CONFIDENCE_MULTIPLIER} SE, no t correction")
print()
print("Resampling cohort per source dataset:")
print(
    unit_table.groupby(["dataset", "unit"], as_index=False)
    .agg(n_units=("n_dataset_units", "first"), tasks=("task", "count"))
    .sort_values(["unit", "dataset"])
    .to_string(index=False)
)

Three things to pin in the docs:

1. **The unit is the patient where the dataset provides one.** BUS-BRA,
   CAMUS, COVID-BLUES, FetalPlanes, MASLD and STMUS all carry a per-image
   `patient` column and are resampled at the patient level. AUL, GBCU,
   MMOTU-2D, OpenKidney, PSFHS and TN3K carry no patient identifier at all,
   so their eleven tasks are necessarily resampled per image. MASLD (11
   patients), COVID-BLUES (20) and CAMUS (50) have small cohorts, which makes
   their bootstrap coarse.
2. **The draw is shared per source dataset.** FetalPlanes is the only case
   where the tasks' cohorts differ -- `fetal_brain` has 741 test patients and
   `fetal` has 1677, sharing 711 -- so the shared draw is taken over the
   union and each task sees the subset of it that its own test split covers.
3. **Use `np.random.default_rng`, not `np.random.seed`.** NumPy guarantees
   the PCG64 stream across versions; the legacy global generator carries no
   such promise. State the NumPy major version used (see above) in the
   appendix anyway.